In [1]:
import pandas as pd
from utils.preprocess import preprocess, process_other

select_lst = [
    '發生月份',
    '天候名稱', '光線名稱',
    '道路類別-第1當事者-名稱', '速限-第1當事者',
    '路面狀況-路面鋪裝名稱', '路面狀況-路面狀態名稱', '路面狀況-路面缺陷名稱',
    '道路障礙-障礙物名稱', '道路障礙-視距品質名稱', '道路障礙-視距名稱',
    '號誌-號誌種類名稱', '號誌-號誌動作名稱',
    '車道劃分設施-分道設施-快車道或一般車道間名稱', '車道劃分設施-分道設施-快慢車道間名稱', '車道劃分設施-分道設施-路面邊線名稱',
    '當事者屬-性-別名稱', '當事者事故發生時年齡',
    '保護裝備名稱', '行動電話或電腦或其他相類功能裝置名稱',
    '肇事逃逸類別名稱-是否肇逃',
    '死亡受傷人數',
    '道路型態大類別名稱', '事故位置大類別名稱',
    '車道劃分設施-分向設施大類別名稱',
    '事故類型及型態大類別名稱', '當事者區分-類別-大類別名稱-車種', '當事者行動狀態大類別名稱',
    '車輛撞擊部位大類別名稱-最初', '車輛撞擊部位大類別名稱-其他',
    '肇因研判大類別名稱-主要',
    '道路型態子類別名稱', '事故位置子類別名稱', '事故類型及型態子類別名稱', '肇因研判子類別名稱-主要',
    '當事者區分-類別-子類別名稱-車種', '當事者行動狀態子類別名稱', '車輛撞擊部位子類別名稱-最初',
    '車輛撞擊部位子類別名稱-其他', '肇因研判子類別名稱-個別',
]

dataA1 = pd.read_csv("./Data/A1.csv")
dataA2 = pd.read_csv("./Data/A2.csv", low_memory=False)

groups = {
    '行人': False, 
    '駕駛': 0.33,
    '汽車': False,
    '機車': 0.6,
}

raw_counts = {}
final_counts = {}

for target, ds in groups.items():
    full_dataA1 = preprocess(dataA1, target=target, lst=select_lst)
    full_dataA2 = preprocess(dataA2, target=target, lst=select_lst)

    raw_counts[target] = (len(full_dataA1), len(full_dataA2))

    mapper_numpy, rbind_data, dummy_data, death, injuried = process_other(
        full_dataA1, full_dataA2, downsample=ds, en=True
    )
    final_counts[target] = rbind_data.shape[0]

print("preprocess() 篩選後、downsample 前的母體筆數 (A1, A2)")
for k, (a1, a2) in raw_counts.items():
    print(f"{k}: A1={a1}, A2={a2}")

print("\n檢查 car + motor 是否真的 <= driver（母體層級，不受 downsample 影響）")
car_a1, car_a2 = raw_counts['汽車']
motor_a1, motor_a2 = raw_counts['機車']
driver_a1, driver_a2 = raw_counts['駕駛']
print(f"A1: car+motor={car_a1+motor_a1} vs driver={driver_a1} " f"-> {'OK' if car_a1+motor_a1 <= driver_a1 else '不一致，母體層級就對不起來!'}")
print(f"A2: car+motor={car_a2+motor_a2} vs driver={driver_a2} " f"-> {'OK' if car_a2+motor_a2 <= driver_a2 else '不一致，母體層級就對不起來!'}")

print("\ndownsample 設定後的最終筆數")
for k, v in final_counts.items():
    print(f"{k}: {v} (downsample={groups[k]})")


/var/folders/w2/_g9w5yys0f171q4qqm469z1h0000gn/T/ipykernel_53230/2590137834.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


['行動電話或電腦或其他相類功能裝置名稱', '當事者區分-類別-大類別名稱-車種', '車輛撞擊部位大類別名稱-最初']
dummy_data: (3143, 210)
46 10354
48 10804
40 9003
35 7878
36 8103
40 9003
48 10804
43 9679
40 9003
48 10804
[]
dummy_data: (95859, 471)
[]
dummy_data: (111956, 453)
43 10898
44 11152
42 10645
37 9377
33 8364
43 10898
54 13686
46 11659
39 9884
42 10645
[]
dummy_data: (107631, 416)
preprocess() 篩選後、downsample 前的母體筆數 (A1, A2)
行人: A1=58, A2=3085
駕駛: A1=1301, A2=292854
汽車: A1=583, A2=111373
機車: A1=716, A2=181475

檢查 car + motor 是否真的 <= driver（母體層級，不受 downsample 影響）
A1: car+motor=1299 vs driver=1301 -> OK
A2: car+motor=292848 vs driver=292854 -> OK

downsample 設定後的最終筆數
行人: 3143 (downsample=False)
駕駛: 95859 (downsample=0.33)
汽車: 111956 (downsample=False)
機車: 107631 (downsample=0.6)
